# SKlearn Pipelines (Polars)

In [1]:
# The difference between sklearn pipelines and transformers is 
# that a pipeline is a sequence of steps. A transformer transforms
# the data, and a pipeline is a sequence of transformers.
# A ColumnTransformer applies multiple transformers to different
# columns of the input data.
import polars as pl
import polars.selectors as cs
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn import set_config
# Custom imports
from get_dataset_polars import get_polars_df
from process_data_polars import tweak_housing

set_config(transform_output='polars')

In [2]:
# Gets Polars dataframe
raw = get_polars_df()

In [3]:
# See what the numeric columns are.
print(tweak_housing(raw).select(cs.numeric()).columns)

['id', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'lat', 'long', 'sqft_living15', 'sqft_lot15']


In [4]:
"""
Imagine comparing number of bedrooms or bathrooms to square feet.
If these are not standardised, some algorithms may pay more attention to square feet
since the numbers are much larger than the no. of rooms.

StandardScaler's fit calculates the mean and standard deviation for each numeric column.
These values are stored internally in the scaler object, std.mean_, std.scale_ etc.

transform produces standardised data with mean that approximates to 0 and varience that 
approximates to 1.

The following codes has additional print statements to work with Python's IDLE.
"""
# Define numeric features
numeric_features = ['bedrooms', 'bathrooms', 'sqft_living']
# Apply StandardScaler
std = StandardScaler()
std_f = std.fit_transform(tweak_housing(raw).select(numeric_features))
print(std_f)

shape: (21_613, 3)
┌───────────┬───────────┬─────────────┐
│ bedrooms  ┆ bathrooms ┆ sqft_living │
│ ---       ┆ ---       ┆ ---         │
│ f64       ┆ f64       ┆ f64         │
╞═══════════╪═══════════╪═════════════╡
│ -0.398737 ┆ -1.447464 ┆ -0.979835   │
│ -0.398737 ┆ 0.175607  ┆ 0.533634    │
│ -1.473959 ┆ -1.447464 ┆ -1.426254   │
│ 0.676485  ┆ 1.149449  ┆ -0.13055    │
│ -0.398737 ┆ -0.149007 ┆ -0.435422   │
│ …         ┆ …         ┆ …           │
│ -0.398737 ┆ 0.500221  ┆ -0.598746   │
│ 0.676485  ┆ 0.500221  ┆ 0.250539    │
│ -1.473959 ┆ -1.772078 ┆ -1.154047   │
│ -0.398737 ┆ 0.500221  ┆ -0.522528   │
│ -1.473959 ┆ -1.772078 ┆ -1.154047   │
└───────────┴───────────┴─────────────┘


In [5]:
# Build pipeline
num_pipeline = Pipeline([
     ('std', StandardScaler())])

# Fit and transform
num_pipe = num_pipeline.fit_transform(
    tweak_housing(raw)
    .select(numeric_features)
)
print(num_pipe)

shape: (21_613, 3)
┌───────────┬───────────┬─────────────┐
│ bedrooms  ┆ bathrooms ┆ sqft_living │
│ ---       ┆ ---       ┆ ---         │
│ f64       ┆ f64       ┆ f64         │
╞═══════════╪═══════════╪═════════════╡
│ -0.398737 ┆ -1.447464 ┆ -0.979835   │
│ -0.398737 ┆ 0.175607  ┆ 0.533634    │
│ -1.473959 ┆ -1.447464 ┆ -1.426254   │
│ 0.676485  ┆ 1.149449  ┆ -0.13055    │
│ -0.398737 ┆ -0.149007 ┆ -0.435422   │
│ …         ┆ …         ┆ …           │
│ -0.398737 ┆ 0.500221  ┆ -0.598746   │
│ 0.676485  ┆ 0.500221  ┆ 0.250539    │
│ -1.473959 ┆ -1.772078 ┆ -1.154047   │
│ -0.398737 ┆ 0.500221  ┆ -0.522528   │
│ -1.473959 ┆ -1.772078 ┆ -1.154047   │
└───────────┴───────────┴─────────────┘


In [6]:
# Add another step
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), # if a value is missing, add the median value
    ('std', StandardScaler())])

# Fit and transform
num_p = num_pipeline.fit_transform(
    tweak_housing(raw)
    .select(numeric_features)
)
print(num_p)

shape: (21_613, 3)
┌───────────┬───────────┬─────────────┐
│ bedrooms  ┆ bathrooms ┆ sqft_living │
│ ---       ┆ ---       ┆ ---         │
│ f64       ┆ f64       ┆ f64         │
╞═══════════╪═══════════╪═════════════╡
│ -0.398737 ┆ -1.447464 ┆ -0.979835   │
│ -0.398737 ┆ 0.175607  ┆ 0.533634    │
│ -1.473959 ┆ -1.447464 ┆ -1.426254   │
│ 0.676485  ┆ 1.149449  ┆ -0.13055    │
│ -0.398737 ┆ -0.149007 ┆ -0.435422   │
│ …         ┆ …         ┆ …           │
│ -0.398737 ┆ 0.500221  ┆ -0.598746   │
│ 0.676485  ┆ 0.500221  ┆ 0.250539    │
│ -1.473959 ┆ -1.772078 ┆ -1.154047   │
│ -0.398737 ┆ 0.500221  ┆ -0.522528   │
│ -1.473959 ┆ -1.772078 ┆ -1.154047   │
└───────────┴───────────┴─────────────┘


In [7]:
"""
If we train it on a dataset that has certain categories and when we try
to do prediction, if it comes across a new category, we ignore it.

max_categories sets maximum no. of columns.
"""
cat_features = ['zipcode']

ohe = OneHotEncoder(handle_unknown='ignore',
                    sparse_output=False, max_categories=10)

# Fit and transform
ohe_f = ohe.fit_transform(
    tweak_housing(raw)
    .select(cat_features)
)
print(ohe_f)

shape: (21_613, 10)
┌──────────────┬──────────────┬──────────────┬─────────────┬───┬─────────────┬─────────────┬─────────────┬─────────────┐
│ zipcode_9802 ┆ zipcode_9803 ┆ zipcode_9803 ┆ zipcode_980 ┆ … ┆ zipcode_981 ┆ zipcode_981 ┆ zipcode_981 ┆ zipcode_inf │
│ 3            ┆ 4            ┆ 8            ┆ 42          ┆   ┆ 15          ┆ 17          ┆ 18          ┆ requent_skl │
│ ---          ┆ ---          ┆ ---          ┆ ---         ┆   ┆ ---         ┆ ---         ┆ ---         ┆ earn        │
│ f64          ┆ f64          ┆ f64          ┆ f64         ┆   ┆ f64         ┆ f64         ┆ f64         ┆ ---         │
│              ┆              ┆              ┆             ┆   ┆             ┆             ┆             ┆ f64         │
╞══════════════╪══════════════╪══════════════╪═════════════╪═══╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 0.0          ┆ 0.0          ┆ 0.0          ┆ 0.0         ┆ … ┆ 0.0         ┆ 0.0         ┆ 0.0         ┆ 1.0         │
│ 0.0       

In [8]:
# Transformer from a function.
tweak_transformer = FunctionTransformer(tweak_housing)
print(tweak_transformer.fit_transform(raw))

shape: (21_613, 21)
┌────────────┬──────────┬──────────┬───────────┬───┬──────────┬───────────────┬────────────┬────────────┐
│ id         ┆ price    ┆ bedrooms ┆ bathrooms ┆ … ┆ long     ┆ sqft_living15 ┆ sqft_lot15 ┆ date       │
│ ---        ┆ ---      ┆ ---      ┆ ---       ┆   ┆ ---      ┆ ---           ┆ ---        ┆ ---        │
│ i64        ┆ f64      ┆ i64      ┆ f64       ┆   ┆ f64      ┆ i64           ┆ i64        ┆ date       │
╞════════════╪══════════╪══════════╪═══════════╪═══╪══════════╪═══════════════╪════════════╪════════════╡
│ 7129300520 ┆ 221900.0 ┆ 3        ┆ 1.0       ┆ … ┆ -122.257 ┆ 1340          ┆ 5650       ┆ 2014-10-13 │
│ 6414100192 ┆ 538000.0 ┆ 3        ┆ 2.25      ┆ … ┆ -122.319 ┆ 1690          ┆ 7639       ┆ 2014-12-09 │
│ 5631500400 ┆ 180000.0 ┆ 2        ┆ 1.0       ┆ … ┆ -122.233 ┆ 2720          ┆ 8062       ┆ 2015-02-25 │
│ 2487200875 ┆ 604000.0 ┆ 4        ┆ 3.0       ┆ … ┆ -122.393 ┆ 1360          ┆ 5000       ┆ 2014-12-09 │
│ 1954400510 ┆ 510000.0 ┆ 

In [9]:
categorical_features = ['zipcode']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

# Column Transformer lets us apply specific transformations to certain columns.
ct = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore',
                              sparse_output=False), categorical_features)])

ct_f = ct.fit_transform(
    tweak_housing(raw)
    .select([*numeric_features, *cat_features])
)
print(ct_f)

shape: (21_613, 73)
┌──────────────┬──────────────┬──────────────┬─────────────┬───┬─────────────┬─────────────┬─────────────┬─────────────┐
│ num__bedroom ┆ num__bathroo ┆ num__sqft_li ┆ cat__zipcod ┆ … ┆ cat__zipcod ┆ cat__zipcod ┆ cat__zipcod ┆ cat__zipcod │
│ s            ┆ ms           ┆ ving         ┆ e_98001     ┆   ┆ e_98178     ┆ e_98188     ┆ e_98198     ┆ e_98199     │
│ ---          ┆ ---          ┆ ---          ┆ ---         ┆   ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│ f64          ┆ f64          ┆ f64          ┆ f64         ┆   ┆ f64         ┆ f64         ┆ f64         ┆ f64         │
╞══════════════╪══════════════╪══════════════╪═════════════╪═══╪═════════════╪═════════════╪═════════════╪═════════════╡
│ -0.398737    ┆ -1.447464    ┆ -0.979835    ┆ 0.0         ┆ … ┆ 1.0         ┆ 0.0         ┆ 0.0         ┆ 0.0         │
│ -0.398737    ┆ 0.175607     ┆ 0.533634     ┆ 0.0         ┆ … ┆ 0.0         ┆ 0.0         ┆ 0.0         ┆ 0.0         │
│ -1.473959 

In [10]:
# Custom transformer that maps a zip code to the average price of that zip code.
class ZipAvgPriceAdder(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
        
    def fit(self, X, y=None):
        # assume X is a polars dataframe
        self.zip_avg_price = (X
                              .group_by('zipcode')
                              .agg(zip_mean=pl.col('price').mean())
        )
        return self
    
    def transform(self, X, y=None):
        return X.join(self.zip_avg_price, on='zipcode')

zip_adder = ZipAvgPriceAdder()
zip_f = zip_adder.fit_transform(raw.select(['zipcode', 'price']))
print(zip_f)

shape: (21_613, 3)
┌─────────┬──────────┬───────────────┐
│ zipcode ┆ price    ┆ zip_mean      │
│ ---     ┆ ---      ┆ ---           │
│ i64     ┆ f64      ┆ f64           │
╞═════════╪══════════╪═══════════════╡
│ 98178   ┆ 221900.0 ┆ 310612.755725 │
│ 98125   ┆ 538000.0 ┆ 469455.770732 │
│ 98028   ┆ 180000.0 ┆ 462480.035336 │
│ 98136   ┆ 604000.0 ┆ 551688.673004 │
│ 98074   ┆ 510000.0 ┆ 685605.77551  │
│ …       ┆ …        ┆ …             │
│ 98103   ┆ 360000.0 ┆ 584919.210963 │
│ 98146   ┆ 400000.0 ┆ 359483.239583 │
│ 98144   ┆ 402101.0 ┆ 594547.650146 │
│ 98027   ┆ 400000.0 ┆ 616990.592233 │
│ 98144   ┆ 325000.0 ┆ 594547.650146 │
└─────────┴──────────┴───────────────┘


### Full example continued at 1d_ii_pipelines.polars.ipynb.